# 10 - SHAP for TF-IDF/LR (exact, linear) and Integrated Gradients for the best encoder

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import models, explain
params = io_utils.read_json(config.SELECTION_LOCK)["models"]["tfidf-lr"]["params"]
r = models.tfidf_lr(X, y, split, seed=0, **params)
vec = r["vectorizer"]
Xtr = vec.transform(X.loc[split == "train", "text"])
Xpos = vec.transform(X.loc[(split == "test") & (y == 1), "text"])
explain.top_terms(r["model"], vec, Xtr, Xpos, k=30)   # association with the model's scores, not causes